# 06 – Feedback for at-risk students

Goal: connect Part 1 and Part 2. For the students the dropout model flags, judge their explanations with the LLM judge (prompt v2) and produce personal feedback.

## Approach
- At-risk students come from `outputs/at_risk_students.csv` (notebook 04: cross-validated risk of 0.4 or more, 180 students).
- Every explanation written by an at-risk student is judged with `judge.run_batch(..., version="v2")`. Calls already made in notebook 05 are reused from the saved results file, and a rerun continues where it stopped.
- Output: `outputs/at_risk_feedback.csv` with student, risk, lesson, the student's text, verdict and the three feedback fields. Five examples are shown below.

## Decisions
| Decision | Choice | Why |
|---|---|---|
| Who gets feedback | All explanations from the 180 flagged students | The goal is help for the students most likely to drop out |
| Prompt | v2 | Better than v1 on the labelled rows and robust to instructions hidden in student text |
| Students with no explanation | No feedback, reported separately | 35 of the 180 flagged students wrote no explanation, so there is nothing to judge; they need a different nudge (for example a reminder) |

In [1]:
#imports and load at-risk students with their explanations
import sys
import importlib
sys.path.append("../src")
import pandas as pd
import judge
importlib.reload(judge)
pd.set_option("display.max_colwidth", None)

at_risk = pd.read_csv("../outputs/at_risk_students.csv")
expl = pd.read_csv("../data/clean/explanations_clean.csv")
todo = expl.merge(at_risk, on="student_id")
print("at-risk students:", len(at_risk))
print("with at least one explanation:", todo.student_id.nunique(), "| explanations to judge:", len(todo))
print("flagged students with no explanation:", len(at_risk) - todo.student_id.nunique())

at-risk students: 180
with at least one explanation: 145 | explanations to judge: 216
flagged students with no explanation: 35


In [2]:
#judge every explanation from an at-risk student (prompt v2)
res = judge.run_batch(todo, version="v2")
print(res.verdict.value_counts().to_dict())

20/216 done
30/216 done
50/216 done
60/216 done
70/216 done
80/216 done
90/216 done
110/216 done
120/216 done
130/216 done
140/216 done
150/216 done
160/216 done
170/216 done
180/216 done
190/216 done
200/216 done
210/216 done
{'partial': 86, 'understood': 77, 'wrong': 53}


In [3]:
#combine with student and risk, and save
out = todo[["student_id", "risk", "explanation_id", "lesson_id", "explanation_text"]].merge(res, on="explanation_id")
out = out.sort_values("risk", ascending=False)
out.to_csv("../outputs/at_risk_feedback.csv", index=False)
print(len(out), "rows saved; judge errors:", (out.verdict == "error").sum())
out.groupby("lesson_id").verdict.value_counts().unstack(fill_value=0)

216 rows saved; judge errors: 0


verdict,partial,understood,wrong
lesson_id,,,
L1,39,39,27
L2,32,25,20
L3,15,13,6


In [4]:
#five example outputs: the highest-risk student for each verdict, then the next highest-risk students
first_per_verdict = out.groupby("verdict").head(1)
rest = out.drop(first_per_verdict.index).head(5 - len(first_per_verdict))
examples = pd.concat([first_per_verdict, rest]).sort_values("risk", ascending=False).head(5)
for r in examples.itertuples():
    print(f"--- {r.student_id} | risk {r.risk} | {r.explanation_id} ({r.lesson_id}) | verdict: {r.verdict}")
    print("Student wrote:", r.explanation_text)
    print("  got right:", r.got_right)
    print("  missed:   ", r.missed)
    print("  hint:     ", r.hint)

--- S323 | risk 0.978 | E0691 (L1) | verdict: understood
Student wrote: Photosynthesis matlab plant sunlight use karke apna food banata hai. Chlorophyll light absorb karta hai, CO2 aur water se glucose banta hai aur oxygen release hoti hai. i think
  got right: You clearly explained how plants use sunlight, carbon dioxide, and water to make glucose and release oxygen.
  missed:    You could also mention specifically where in the plant this process mainly takes place.
  hint:      Do you remember which part of the plant plays the biggest role in capturing light and making food?
--- S482 | risk 0.962 | E0167 (L1) | verdict: partial
Student wrote: Photosynthesis is the process where plants make glucose. It needs chlorophyll which makes leaves green. hope this is right!
  got right: You correctly explained that plants make glucose during photosynthesis using chlorophyll.
  missed:    You missed the inputs and outputs, such as carbon dioxide, water, and oxygen.
  hint:      What do plants t

## Findings

- **Coverage:** 216 explanations from 145 of the 180 at-risk students were judged, with 0 judge errors. The other 35 flagged students wrote no explanation, so the feedback tool cannot reach them; they need a different nudge.
- **Verdicts:** 86 `partial` (40%), 77 `understood` (36%), 53 `wrong` (25%). The pattern is similar across the three lessons.
- **Not every at-risk student is struggling with the content.** About one in three flagged students wrote an explanation the judge rates `understood`. For them the problem is probably engagement, not understanding, so a content hint is less useful than an encouraging or reminding message. The two signals together (risk score plus verdict) can pick the right kind of follow-up.
- **Examples:** the five outputs above show a `wrong` answer that contradicts the lesson (S083, S484), a `partial` answer that is missing the inputs and outputs (S482), and an `understood` answer written in Hinglish (S323) that the judge handled correctly. This is one example, not a measurement of Indian-language quality.
- **Limit:** `missed` sometimes names the answer (for example S484/E0335 lists sunlight, carbon dioxide and water), so the "one hint, not the full answer" rule is only partly met. The verdicts here are not checked against human labels (only the 60 labelled rows are); they inherit the accuracy limits from notebook 05.